<div align="center">

# BANK MARKETING SUBSCRIPTION PREDICTION

### Machine Learning Project-Based Learning (PBL)

**A Data-Driven Approach to Predicting Term Deposit Subscription**

---

**Submitted by:** Ajeet Sherkar  
**Program:** B.Tech — Artificial Intelligence & Machine Learning  
**University:** Sanjivani University, Kopargaon, Maharashtra, India  
**Academic Year:** 2026–2027

---

### Technology Stack

Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn  
Imbalanced-learn · Gradient Boosting · Streamlit · Jupyter Notebook

</div>

## Abstract

The Bank Marketing Subscription Prediction project applies supervised machine
learning to identify customers who may subscribe to a bank term deposit.
The project uses the UCI Bank Marketing dataset and frames the task as a
binary classification problem.

The workflow includes business understanding, dataset inspection, preprocessing,
exploratory data analysis, feature engineering, model comparison, evaluation,
business interpretation, and deployment through a Streamlit application.

The final model uses a Gradient Boosting classifier in a preprocessing pipeline.
Because the target classes are imbalanced, model evaluation considers precision,
recall, F1-score, ROC-AUC, and PR-AUC in addition to accuracy. A decision
threshold of 0.7419 was selected during validation to balance contact volume
and subscription detection according to the project's threshold analysis.

This notebook consolidates the completed project work and supporting artifacts.

## Table of Contents

1. Problem Definition & Business Understanding
2. Dataset & Data Understanding
3. Data Preprocessing & Cleaning
4. Exploratory Data Analysis
5. Feature Engineering & Selection
6. Model Selection & Implementation
7. Model Evaluation & Validation
8. Business Insights & Recommendations
9. Results, Limitations & Conclusion
10. Deployment & Application
11. References and Project Artifacts

## Rubric-to-Notebook Mapping

| Evaluation component | Notebook section |
|---|---|
| Business understanding | 1 |
| Dataset understanding | 2 |
| Data preprocessing | 3 |
| Exploratory data analysis | 4 |
| Feature engineering | 5 |
| Model selection and implementation | 6 |
| Evaluation and validation | 7 |
| Business insights | 8 |
| Deployment | 10 |
| Documentation and conclusion | 9–11 |

*The mapping follows the project rubric discussed during planning. Confirm exact
marks and wording against the official assignment document before submission.*

# 1. Problem Definition & Business Understanding

## 1.1 Problem Statement

Banks use marketing campaigns to encourage customers to subscribe to term
deposit products. Contacting every customer can be costly and inefficient.
The objective is to develop a machine learning model that estimates the
likelihood of subscription using customer and campaign information available
before a marketing call.

## 1.2 Objectives

- Understand the business problem and dataset.
- Prepare data for supervised binary classification.
- Explore customer and campaign patterns.
- Compare multiple classification algorithms.
- Evaluate the selected model using imbalance-aware metrics.
- Interpret model predictions for marketing decision support.
- Demonstrate the model through a Streamlit application.

## 1.3 Business Impact

A predictive model can help marketing teams prioritize customer outreach,
allocate campaign resources, and investigate customer segments with higher
predicted subscription likelihood. Predictions are decision-support estimates,
not guarantees of customer behavior.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

warnings.filterwarnings("ignore")

# Resolve the project root reliably, whether the notebook runs from
# the project directory or from inside the notebooks directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]

PROJECT_ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "data" / "raw" / "bank.csv").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/raw/bank.csv. "
        f"Current working directory: {cwd}"
    )

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
IMAGES_DIR = PROJECT_ROOT / "images"
MODEL_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("Reports directory exists:", REPORTS_DIR.exists())
print("Images directory exists:", IMAGES_DIR.exists())
print("Model directory exists:", MODEL_DIR.exists())

Project root: /Users/ajeetsherkar/Bank-Marketing-ML-PBL
Data directory exists: True
Reports directory exists: True
Images directory exists: True
Model directory exists: True


# 2. Dataset & Data Understanding

## 2.1 Dataset Overview

The project uses the Bank Marketing dataset. The labeled training data contains
4,521 observations and 17 original columns. The target variable is `y`, which
indicates whether the customer subscribed to a term deposit (`yes` or `no`).

The separate test dataset contains customer records without target labels.
Therefore, it can be used for generating predictions, but not for calculating
test accuracy or other label-dependent evaluation metrics.

In [2]:
# The source bank.csv file uses a semicolon delimiter.
data_path = DATA_DIR / "raw" / "bank.csv"
df = pd.read_csv(data_path, sep=";")

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

Dataset shape: (4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


<class 'pandas.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        4521 non-null   int64
 1   job        4521 non-null   str  
 2   marital    4521 non-null   str  
 3   education  4521 non-null   str  
 4   default    4521 non-null   str  
 5   balance    4521 non-null   int64
 6   housing    4521 non-null   str  
 7   loan       4521 non-null   str  
 8   contact    4521 non-null   str  
 9   day        4521 non-null   int64
 10  month      4521 non-null   str  
 11  duration   4521 non-null   int64
 12  campaign   4521 non-null   int64
 13  pdays      4521 non-null   int64
 14  previous   4521 non-null   int64
 15  poutcome   4521 non-null   str  
 16  y          4521 non-null   str  
dtypes: int64(7), str(10)
memory usage: 826.8 KB


None

In [3]:
print("Column names:")
display(pd.DataFrame({"Column": df.columns}))

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nTarget distribution:")
target_counts = df["y"].value_counts(dropna=False)
target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": (target_counts / len(df) * 100).round(2)
})
display(target_summary)

Column names:


,Column
0,age
1,job
2,marital
3,education
4,default
5,balance
6,housing
7,loan
8,contact
9,day



Data types:


,Data Type
age,int64
job,str
marital,str
education,str
default,str
balance,int64
housing,str
loan,str
contact,str
day,int64



Target distribution:


,Count,Percentage
y,,
no,4000,88.48
yes,521,11.52


## 2.2 Data Dictionary

The project's data dictionary is maintained in `reports/data_dictionary.md`.
The source file is displayed below when available.

In [4]:
dictionary_path = REPORTS_DIR / "data_dictionary.md"
if dictionary_path.exists():
    print(dictionary_path.read_text(encoding="utf-8"))
else:
    print("Data dictionary file was not found.")

# Bank Marketing Dataset – Data Dictionary

## Dataset Overview

- Dataset: Bank Marketing
- Source: UCI Bank Marketing dataset documentation
- Records: 4,521
- Columns: 17
- Target variable: y
- Problem type: Binary Classification

## Feature Descriptions

| Feature | Description | Data Type | Role |
|---|---|---|---|
| age | Age of the bank client | Numerical | Predictor |
| job | Type of job | Categorical | Predictor |
| marital | Marital status; divorced includes divorced or widowed | Categorical | Predictor |
| education | Education level | Categorical | Predictor |
| default | Whether the client has credit in default | Categorical | Predictor |
| balance | Average yearly balance in euros | Numerical | Predictor |
| housing | Whether the client has a housing loan | Categorical | Predictor |
| loan | Whether the client has a personal loan | Categorical | Predictor |
| contact | Communication type used for the last contact | Categorical | Predictor |
| day | Last contact day of the 

# 3. Data Preprocessing & Cleaning

Preprocessing is required to ensure that the data is suitable for model
training. The project pipeline handles numerical and categorical features
separately.

Key considerations include:

- Inspecting missing values and duplicate records.
- Preserving the binary target definition.
- Scaling numerical features.
- Encoding categorical features.
- Handling class imbalance with SMOTENC within the training pipeline.
- Avoiding information leakage from variables unavailable at prediction time.

### Target encoding

The model uses `no = 0` and `yes = 1`.

### Leakage prevention

The feature `duration` is excluded because it represents the length of a
marketing call. It is not known before the call begins and would not be
available for a pre-call prediction.

In [5]:
print("Missing values by column:")
display(df.isna().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", int(df.duplicated().sum()))

print("\nTarget labels:")
display(df["y"].value_counts())

Missing values by column:


,Missing Values
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0



Duplicate rows: 0

Target labels:


y
no     4000
yes     521
Name: count, dtype: int64

## 3.1 Preprocessing Documentation

The detailed preprocessing decisions and implementation are documented in the
existing project report. The saved model pipeline also encapsulates its
preprocessing steps.

In [6]:
preprocessing_report = REPORTS_DIR / "model_packaging_report.md"
if preprocessing_report.exists():
    print(preprocessing_report.read_text(encoding="utf-8"))
else:
    print("Packaging report not found.")

# Phase 9 — Model Packaging and Reproducibility

## 1. Objective
Package the selected machine learning model, preserve its metadata,
provide reusable training and evaluation scripts, and generate
predictions on unseen customer records.

## 2. Selected Model
- Model: Gradient Boosting Classifier
- Preprocessing: SMOTENC, StandardScaler and OneHotEncoder
- Input features: 15
- Excluded feature: duration
- Decision threshold: 0.7419

## 3. Model Artifacts
- models/best_model.joblib
- models/model_metadata.json

The trained pipeline and its configuration are saved for reuse.

## 4. Reproducibility Scripts
- src/train.py: trains and saves the model and metadata.
- src/evaluate.py: evaluates a saved model on labelled data.
- src/predict.py: generates subscription predictions for new customers.

## 5. Evaluation
Validation results from the training workflow at threshold 0.50:

- Accuracy: 0.7282
- Precision: 0.2205
- Recall: 0.5385
- F1-score: 0.3128
- ROC-AUC: 0.7171
- PR-AUC: 0.3211

At thr

# 4. Exploratory Data Analysis

Exploratory Data Analysis (EDA) helps identify class imbalance, customer
characteristics, campaign patterns, and possible relationships with the
subscription target.

The figures below are the saved visualizations generated during the original
project analysis.

### Target distribution

![Target distribution](../images/eda/01_target_distribution.png)

### Age distribution

![Age distribution](../images/eda/02_age_distribution.png)

### Numerical feature boxplots

![Numerical feature boxplots](../images/eda/03_numerical_feature_boxplots.png)

### Numerical correlation heatmap

![Numerical correlation heatmap](../images/eda/04_numerical_correlation_heatmap.png)

### Job distribution

![Job distribution](../images/eda/05_job_distribution.png)

### Subscription rate by job

![Subscription rate by job](../images/eda/06_subscription_rate_by_job.png)

### Subscription rate by campaign

![Subscription rate by campaign](../images/eda/07_subscription_rate_by_campaign.png)

### Subscription rate by education

![Subscription rate by education](../images/eda/07_subscription_rate_by_education.png)

### Subscription rate by housing loan

![Subscription rate by housing loan](../images/eda/07_subscription_rate_by_housing.png)

### Subscription rate by personal loan

![Subscription rate by personal loan](../images/eda/07_subscription_rate_by_loan.png)

### Subscription rate by marital status

![Subscription rate by marital status](../images/eda/07_subscription_rate_by_marital.png)

### Subscription rate by month

![Subscription rate by month](../images/eda/07_subscription_rate_by_month.png)

### Subscription rate by previous outcome

![Subscription rate by previous outcome](../images/eda/07_subscription_rate_by_poutcome.png)

### Numerical features versus subscription

![Numerical features versus subscription](../images/eda/08_numerical_features_vs_subscription.png)

## 4.1 EDA Interpretation

The target distribution indicates that the dataset is imbalanced, with
subscription responses (`yes`) forming a minority class. Consequently, a model
that predicts only the majority class could achieve misleadingly high accuracy.

The saved visualizations provide evidence for investigating customer
demographics, campaign frequency, contact month, previous campaign outcome,
and other variables. These patterns should be interpreted as associations
within the dataset rather than causal effects.

In [7]:
eda_report = REPORTS_DIR / "eda_report.md"
if eda_report.exists():
    print(eda_report.read_text(encoding="utf-8"))
else:
    print("EDA report not found.")

# Exploratory Data Analysis Report
## Bank Marketing Subscription Prediction

## 1. Project Overview

This report presents the Exploratory Data Analysis (EDA) performed on the Bank Marketing dataset. The objective is to understand customer characteristics, identify patterns associated with term-deposit subscriptions, and derive insights that can guide feature engineering and predictive model development.

The target variable is `y`:
- `yes`: Customer subscribed to a term deposit.
- `no`: Customer did not subscribe.

The analysis uses the training dataset containing 4,521 records and 17 columns.

## 2. Target Variable Analysis

The target distribution is:

| Subscription Outcome | Customers | Percentage |
|---|---:|---:|
| No | 4,000 | 88.48% |
| Yes | 521 | 11.52% |
| Total | 4,521 | 100% |

The dataset is imbalanced, with substantially more non-subscribers than subscribers.

**Figure 1: Target Distribution**

![Target Distribution](../images/eda/01_target_distribution.png)

**Business

# 5. Feature Engineering & Selection

Feature engineering prepares the model inputs and supports the selection of
useful predictors.

The final model uses the following feature groups:

**Numerical features:** `age`, `balance`, `day`, `campaign`, `pdays`, `previous`

**Categorical features:** `job`, `marital`, `education`, `default`, `housing`,
`loan`, `contact`, `month`, `poutcome`

The `duration` feature is intentionally excluded due to leakage risk in
pre-call prediction.

### Feature correlation

![Feature correlation](../images/feature_engineering/01_feature_correlation.png)

### Mutual information

![Mutual information](../images/feature_engineering/02_mutual_information.png)

### Random Forest feature importance

![Random Forest feature importance](../images/feature_engineering/03_random_forest_importance.png)

### Permutation importance

![Permutation importance](../images/feature_engineering/04_permutation_importance.png)

In [8]:
feature_report = REPORTS_DIR / "feature_engineering_report.md"
if feature_report.exists():
    print(feature_report.read_text(encoding="utf-8"))
else:
    print("Feature engineering report not found.")

# Phase 5: Feature Engineering and Feature Selection

## Project
Bank Marketing Subscription Prediction

## 1. Objective

Investigate engineered features and apply feature-selection techniques to identify useful predictors of bank marketing subscription.

## 2. Dataset

- Observations: 4,521
- Original columns: 17
- Target: y
- Training records: 3,616
- Validation records: 905
- Training-validation split: Stratified 80:20
- Random state: 42
- Duration excluded from predictive features due to potential target leakage.

## 3. Engineered Features

Five candidate features were investigated:

1. age_group
2. balance_group
3. has_previous_contact
4. total_contacts
5. campaign_intensity

Balance thresholds were learned from training data only.

Original features were retained for comparison.

## 4. Feature Selection Techniques

The following techniques were investigated:

- Pearson correlation analysis
- Mutual Information
- Random Forest impurity-based importance
- Permutation importance usi

# 6. Model Selection & Implementation

Several classification algorithms were compared during model development.
The final selected algorithm is Gradient Boosting.

The model pipeline uses:

1. SMOTENC for class-imbalance handling.
2. ColumnTransformer for numerical and categorical preprocessing.
3. StandardScaler for numerical features.
4. OneHotEncoder for categorical features.
5. GradientBoostingClassifier for binary classification.

The saved model artifact is `models/best_model.joblib`. This notebook loads the
artifact for inspection and does not retrain or overwrite it.

### Logistic Regression confusion matrix

![Logistic Regression confusion matrix](../images/model_development/01_logistic_confusion_matrix.png)

### Logistic Regression ROC and PR curves

![Logistic Regression ROC and PR curves](../images/model_development/02_logistic_roc_pr_curves.png)

### Tree model confusion matrices

![Tree model confusion matrices](../images/model_development/03_tree_models_confusion_matrices.png)

### Model ROC and PR comparison

![Model ROC and PR comparison](../images/model_development/04_model_roc_pr_comparison.png)

### Random Forest feature importance

![Random Forest feature importance](../images/model_development/05_random_forest_feature_importance.png)

### Five-model confusion matrices

![Five-model confusion matrices](../images/model_development/06_five_model_confusion_matrices.png)

### Five-model ROC and PR comparison

![Five-model ROC and PR comparison](../images/model_development/07_five_model_roc_pr_comparison.png)

In [9]:
comparison_path = REPORTS_DIR / "model_comparison.csv"

if comparison_path.exists():
    model_comparison = pd.read_csv(comparison_path)
    print("Model comparison table:")
    display(model_comparison.head(10))
else:
    print("Model comparison CSV not found.")

Model comparison table:


,Model,Accuracy_Baseline,Precision_Baseline,Recall_Baseline,F1-Score_Baseline,ROC-AUC_Baseline,PR-AUC_Baseline,Accuracy_Tuned,Precision_Tuned,Recall_Tuned,...,True Negatives,False Positives,False Negatives,True Positives,Customers Contacted,Potential Subscribers Identified,Subscribers Missed,Precision,Recall,Final Selection
0,Logistic Regression,0.670718,0.189103,0.567308,0.283654,0.688046,0.289854,0.670718,0.189103,0.567308,...,548,253,45,59,312,59,45,0.189103,0.567308,Not Selected
1,Decision Tree,0.771271,0.227513,0.413462,0.293515,0.615595,0.161471,0.651934,0.175385,0.548077,...,533,268,47,57,325,57,47,0.175385,0.548077,Not Selected
2,Random Forest,0.816575,0.268657,0.346154,0.302521,0.697908,0.267279,0.696133,0.208191,0.586538,...,569,232,43,61,293,61,43,0.208191,0.586538,Not Selected
3,SVM,0.750276,0.220183,0.461538,0.298137,0.687926,0.237767,0.588950,0.176329,0.701923,...,460,341,31,73,414,73,31,0.176329,0.701923,Not Selected
4,Gradient Boosting,0.740331,0.228216,0.528846,0.318841,0.722126,0.310178,0.728177,0.220472,0.538462,...,603,198,48,56,254,56,48,0.220472,0.538462,Selected


In [10]:
metadata_path = MODEL_DIR / "model_metadata.json"

if metadata_path.exists():
    model_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    display(pd.Series(model_metadata, name="Model Metadata").to_frame())
else:
    print("Model metadata JSON not found.")

,Model Metadata
project,Bank Marketing Subscription Prediction
model_name,Gradient Boosting
model_type,GradientBoostingClassifier
random_state,42
decision_threshold,0.7419
threshold_selection,Selected using validation F1 analysis in Phase 7
features,"[age, balance, day, campaign, pdays, previous,..."
numerical_features,"[age, balance, day, campaign, pdays, previous]"
categorical_features,"[job, marital, education, default, housing, lo..."
excluded_features,[duration]


# 7. Model Evaluation & Validation

Evaluation is performed on the held-out validation split. Since the target is
imbalanced, accuracy is reported alongside precision, recall, F1-score,
ROC-AUC, and PR-AUC.

The final selected threshold is **0.7419**. Threshold selection changes the
classification decision, but does not change the underlying ranking metrics
ROC-AUC and PR-AUC.

The saved model artifact is subsequently retrained on the full labeled dataset
for deployment. Therefore, held-out validation metrics should not be confused
with evaluation metrics on the final full-data artifact.

### Selected model evaluation

![Selected model evaluation](../images/evaluation/01_selected_model_evaluation.png)

### Threshold performance

![Threshold performance](../images/evaluation/02_threshold_performance.png)

### Cross-validation stability

![Cross-validation stability](../images/evaluation/03_cross_validation_stability.png)

### Five-model comparison

![Five-model comparison](../images/evaluation/04_five_model_comparison.png)

In [11]:
evaluation_path = REPORTS_DIR / "evaluation_results.json"

if evaluation_path.exists():
    evaluation_results = json.loads(evaluation_path.read_text(encoding="utf-8"))
    print(json.dumps(evaluation_results, indent=2))
else:
    print("Evaluation results JSON not found.")

{
  "project": "Bank Marketing Subscription Prediction",
  "model_name": "Gradient Boosting",
  "evaluation_type": "Stratified validation split",
  "random_state": 42,
  "validation_size": 905,
  "validation_positive_cases": 104,
  "validation_negative_cases": 801,
  "training_time_seconds": 0.9334803749989078,
  "threshold_default": 0.5,
  "threshold_selected": 0.7419,
  "metrics_at_default_threshold": {
    "accuracy": 0.7281767955801105,
    "precision": 0.2204724409448819,
    "recall": 0.5384615384615384,
    "f1_score": 0.3128491620111732,
    "roc_auc": 0.7170844137136273,
    "pr_auc": 0.3211330619245117,
    "true_negatives": 603,
    "false_positives": 198,
    "false_negatives": 48,
    "true_positives": 56,
    "customers_contacted": 254
  },
  "metrics_at_selected_threshold": {
    "accuracy": 0.8806629834254144,
    "precision": 0.46551724137931033,
    "recall": 0.25961538461538464,
    "f1_score": 0.3333333333333333,
    "roc_auc": 0.7170844137136273,
    "pr_auc": 0.32

## 7.1 Evaluation Interpretation

At the default threshold of 0.50, the validation results recorded in the project
are:

| Metric | Value |
|---|---:|
| Accuracy | 0.7282 |
| Precision | 0.2205 |
| Recall | 0.5385 |
| F1-score | 0.3128 |
| ROC-AUC | 0.7171 |
| PR-AUC | 0.3211 |

At the selected threshold of 0.7419:

| Metric | Value |
|---|---:|
| Accuracy | 0.8807 |
| Precision | 0.4655 |
| Recall | 0.2596 |
| F1-score | 0.3333 |
| ROC-AUC | 0.7171 |
| PR-AUC | 0.3211 |

The selected threshold reduces predicted positive contacts and improves precision,
but lowers recall. This means fewer customers are flagged, while some actual
subscribers are missed. The suitable operating threshold depends on the bank's
campaign costs and business priorities.

# 8. Business Insights & Recommendations

Model outputs can help marketing teams prioritize outreach and understand
customer-level prediction patterns. Feature importance can provide insight into
which variables influenced model predictions, but it does not establish
causation.

The project includes saved feature importance and predicted probability
analysis.

### Feature importance

![Feature importance](../images/business_insights/01_feature_importance.png)

### Predicted probability analysis

![Predicted probability analysis](../images/business_insights/02_predicted_probability_analysis.png)

In [12]:
for report_name in ["business_insights_report.md", "predicted_probability_analysis.csv"]:
    report_path = REPORTS_DIR / report_name
    print(f"\n========== {report_name} ==========")
    if report_path.exists() and report_path.suffix == ".md":
        print(report_path.read_text(encoding="utf-8"))
    elif report_path.exists() and report_path.suffix == ".csv":
        display(pd.read_csv(report_path).head(15))
    else:
        print("Artifact not found.")


========== business_insights_report.md ==========
# Phase 8 — Business Insights and Recommendations

## 1. Objective

Translate the machine learning findings into practical recommendations for improving bank marketing campaigns and identifying potential term-deposit subscribers.

## 2. Dataset Overview

- Total customer records: 4,521
- Non-subscribers: 4,000 (88.48%)
- Subscribers: 521 (11.52%)
- Missing values: 0
- Duplicate records: 0

The dataset is imbalanced, so marketing decisions should not rely on accuracy alone.

## 3. Customer Segment Analysis

### Subscription by Job

| Job | Customers | Subscribers | Subscription Rate |
|---|---:|---:|---:|
| Retired | 230 | 54 | 23.48% |
| Student | 84 | 19 | 22.62% |
| Management | 969 | 131 | 13.52% |
| Blue-collar | 946 | 69 | 7.29% |

Retired and student customers show relatively high subscription rates. These segments may deserve further targeted campaign analysis.

### Subscription by Marital Status

| Marital Status | Customers | 

,Probability_Group,Total_Customers,Actual_Subscribers,Average_Predicted_Probability,Actual_Subscription_Rate
0,0-20%,372,17,0.104329,4.569892
1,20-40%,162,16,0.292468,9.876543
2,40-60%,210,29,0.495359,13.809524
3,60-80%,126,22,0.679876,17.460317
4,80-100%,35,20,0.861724,57.142857


## 8.1 Recommendations

- Use model scores as one input to campaign prioritization.
- Consider the trade-off between precision, recall, contact cost, and missed opportunities.
- Monitor performance as customer behavior and campaign conditions change.
- Avoid treating feature importance as causal evidence.
- Review fairness, privacy, and customer-contact policies before operational use.

# 9. Results, Limitations & Conclusion

## 9.1 Conclusion

This project demonstrates an end-to-end machine learning workflow for bank
term-deposit subscription prediction. It combines dataset analysis,
preprocessing, feature engineering, model comparison, validation, business
interpretation, and an application interface.

Gradient Boosting was selected as the final model in the project. The selected
decision threshold reflects a trade-off between identifying subscribers and
limiting the number of customers flagged for outreach.

## 9.2 Limitations

- The labeled dataset is relatively small and represents a specific marketing context.
- The target class is imbalanced.
- The threshold depends on business costs and campaign priorities.
- The separate test dataset has no target labels, so test accuracy cannot be calculated.
- Model predictions are estimates and do not guarantee subscription.
- Historical relationships may change over time.
- Feature importance indicates model association, not causation.

## 9.3 Future Scope

- Evaluate the model on newly labeled campaign data.
- Monitor model calibration and prediction drift.
- Conduct cost-sensitive threshold analysis with explicit campaign costs.
- Investigate fairness and responsible customer targeting.
- Improve application monitoring and deployment practices.

# 10. Deployment & Application

The project includes a Streamlit application that exposes model functionality
through an interactive interface.

The application contains:

- Home
- Customer Prediction
- Batch Prediction
- Model Performance
- Business Insights

The interface is intended as a decision-support demonstration.

### Application Home

![Application Home](../images/application/01_home.png)

### Customer Prediction

![Customer Prediction](../images/application/02_customer_prediction.png)

### Batch Prediction

![Batch Prediction](../images/application/03_batch_prediction.png)

### Model Performance

![Model Performance](../images/application/04_model_performance.png)

### Business Insights

![Business Insights](../images/application/05_business_insights.png)

## 10.1 Application Workflow

```text
Customer / Batch Input
        |
        v
Input Validation
        |
        v
Saved Model Pipeline
        |
        v
Subscription Probability
        |
        v
Selected Threshold (0.7419)
        |
        v
Prediction + Decision-Support Output
```

The model is loaded from the saved artifact. The application does not guarantee
that a customer will subscribe.

# 11. References and Project Artifacts

## Project Artifacts

- Dataset: `data/raw/bank.csv`
- Unlabeled test data: `data/raw/test.csv`
- Saved model: `models/best_model.joblib`
- Model metadata: `models/model_metadata.json`
- Evaluation results: `reports/evaluation_results.json`
- Model comparison: `reports/model_comparison.csv`
- Detailed project report: `reports/project_report.pdf`
- Project source notebooks: `notebooks/01_dataset_understanding.ipynb` through
  `notebooks/07_business_insights.ipynb`

## Dataset Reference

Bank Marketing dataset, UCI Machine Learning Repository. Refer to the project
data documentation and original dataset notes included in `data/raw/`.

## Reproducibility Note

This notebook consolidates existing project results and artifacts. It does not
retrain or overwrite the final model. Validation metrics refer to the held-out
validation split recorded by the project workflow. The unlabeled test dataset
is used for prediction only; label-dependent test metrics cannot be calculated
without ground-truth labels.